# Sensor Fusion — One Big EKF for Lidar + Radar

In this notebook you'll build **one** Extended Kalman Filter that tracks a vehicle by fusing **lidar** and **radar** — the core estimator behind every self-driving perception stack.

**The big idea:** it's *one* filter with *one* state `[px, py, vx, vy]`. Lidar and radar don't each get their own filter — they take turns correcting the same shared belief. That's what "sensor fusion" actually means.

There is **no object detector** here. Detection is a different course. Here the measurements are given, and we focus 100% on *estimation*.

| Sensor | Gives us | Model | Note |
|---|---|---|---|
| Lidar | `(x, y)` | linear | precise position |
| Radar | `(range, bearing, range-rate)` | **non-linear** | velocity + sees through occlusion |

The radar's polar model is non-linear → we linearize it with a **Jacobian** → that's the *Extended* in EKF.


## 0. Setup (Colab)

Clone the course repo and install the package. Zero local setup.


In [ ]:
# On Colab, uncomment:
# !git clone https://github.com/Jeremy26/kalman_filters_course.git
# %cd kalman_filters_course/ekf_sensor_fusion
# !pip install -e '.[viz]' -q

import sys; sys.path.insert(0, 'src')   # if running from the repo without installing
import numpy as np
import matplotlib.pyplot as plt


## 1. Generate the data

A realistic curved trajectory, sensed by a noisy lidar and radar. Midway, the **lidar is occluded** (the target passes behind an obstacle) — remember this, it's the whole point later.


In [ ]:
!python data/generate_dataset.py --out data/fusion_log.txt

from kf_fusion import read_log
measurements = read_log('data/fusion_log.txt')
print(f'{len(measurements)} measurements')
print('first lidar:', next(m for m in measurements if m.sensor=='lidar').z)
print('first radar:', next(m for m in measurements if m.sensor=='radar').z)


## 2. The filter, one step at a time

Every measurement triggers **predict → update**:

- **predict**: push the state forward with the constant-velocity model `F`, and grow the covariance `P` by the process noise `Q`.
- **update**: correct with the sensor — linear `H` for lidar, the Jacobian `Hj` for radar.

The code is in `kf_fusion/ekf.py` and `kf_fusion/models.py` — read them, they're short and un-magical.


In [ ]:
from kf_fusion import EKF
from kf_fusion import models

# The radar model is non-linear. Here's h(x) (polar) and its Jacobian:
x = np.array([4.0, 3.0, 2.0, -1.0])
print('h(x)  =', models.radar_measurement(x))      # [range, bearing, range-rate]
print('Hj    =\n', np.round(models.radar_jacobian(x), 3))


## 3. Run the fusion

`run_fusion` walks the stream and returns the filtered track + metrics.


In [ ]:
from kf_fusion import run_fusion

result = run_fusion(measurements)
s = result.summary()
print(f"Position RMSE: {s['rmse_pos']:.3f} m")
print(f"NIS under 95% bound — lidar {s['nis_lidar_below_95']:.2f}, radar {s['nis_radar_below_95']:.2f}  (target ~0.95)")


## 4. See it

Estimate vs. ground truth. Notice the estimate stays glued to the truth even through the occluded stretch.


In [ ]:
est, gt = result.estimates, result.ground_truth
plt.figure(figsize=(8,6))
plt.plot(gt[:,0], gt[:,1], 'g-', lw=2, label='ground truth')
plt.plot(est[:,0], est[:,1], 'b--', lw=2, label='EKF estimate')
plt.axis('equal'); plt.legend(); plt.title('Fused lidar + radar track'); plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.show()


## 5. Why fuse? The ablation

Run the **same timeline** three ways. A disabled sensor still advances time (predict-only) — so lidar-only must *coast* through its occlusion instead of cheating by skipping those frames.


In [ ]:
fused      = run_fusion(measurements).summary()
lidar_only = run_fusion(measurements, use_radar=False).summary()
radar_only = run_fusion(measurements, use_lidar=False).summary()

for name, r in [('FUSED', fused), ('LIDAR only', lidar_only), ('RADAR only', radar_only)]:
    print(f"{name:11s} pos RMSE = {r['rmse_pos']:.3f} m")

print('\nFusion wins because the sensors cover each other\'s failure modes —')
print('lidar is blind during the occlusion, radar is always noisy in bearing.')


## 6. Export to Foxglove

Write an `.mcap` recording, download it, and open it at [app.foxglove.dev](https://app.foxglove.dev) with `layouts/ekf_fusion.json`.

In Foxglove you can *scrub the timeline* and watch the **covariance ellipse breathe** — growing while the filter dead-reckons through the occlusion, snapping tight the instant a measurement lands. That intuition is the whole course in one view.


In [ ]:
run_fusion(measurements, mcap_path='outputs/ekf_fusion.mcap')
print('Wrote outputs/ekf_fusion.mcap')

# On Colab:
# from google.colab import files; files.download('outputs/ekf_fusion.mcap')


## 7. Your turn

1. **Tune the process noise.** Change `run_fusion(..., noise_ax=, noise_ay=)`. Watch the NIS: too-small noise → over-confident filter (NIS blows past the bound); too-large → sluggish. Find the value that keeps ~95% of NIS under the line.
2. **Move the occlusion** in `generate_dataset.py` onto the sharpest part of the curve. Does lidar-only fail worse? Why?
3. **Break the radar Jacobian** (drop the range-rate row) and watch velocity estimation collapse.

> **Next course:** run *N* of these filters at once and you need to decide which measurement belongs to which track — that's **data association**, and it's where multi-object tracking begins.
